# Configurable Project Runs & Study Workspace Management

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

Reproducible scientific analysis requires separating source code, input trajectories, parameter specifications, and generated artifacts.

MolScope organizes investigations into structured **Study Workspaces**:
```text
my-study/
├── project.json              <-- Validated declarative project specification
├── inputs/                   <-- PDB structures, topologies, and trajectories
├── preparation/              <-- Protonation, imaging, and parameter logs
├── simulations/              <-- OpenMM execution segments & checkpoints
└── runs/
    └── run-001/              <-- Fully auditable, stage-grouped run artifacts
        ├── resolved_project.json
        ├── provenance.json
        ├── features/
        ├── discovery/
        ├── predictive/
        └── report.html
```


## 1. Initializing a Study Workspace via Python API


In [ ]:
from pathlib import Path
from molscope.core.workspace import initialize_study, begin_run

workspace_dir = Path("results/workspace_demo")
workspace_dir.mkdir(parents=True, exist_ok=True)

# Initialize a managed study workspace
study_path = initialize_study(destination=workspace_dir / "villin_study", project_id="villin_study")
print(f"Study workspace initialized at: {study_path}")

# Check directory structure
for child in sorted(study_path.iterdir()):
    print(f"  - {child.name}")


## 2. Declarative `project.json` Specification

A MolScope study is governed by a validated `project.json` file.
It declares systems, ensembles, replicas, metadata conditions, imaging policies, and feature extraction plugins.


In [ ]:
from molscope.core.project import (
    ProjectSpec, SystemSpec, EnsembleSpec, ReplicaSpec, LabelSpec, PredictionTask, AnalysisSpec
)
import json

# Locate test fixture
pdb_path = Path("tests/fixtures/1A22.pdb")
if not pdb_path.is_file():
    pdb_path = Path("../tests/fixtures/1A22.pdb")

# Construct declarative specification
project = ProjectSpec(
    project_id="workspace_example",
    systems=(
        SystemSpec(
            system_id="hgh_receptor",
            topology=pdb_path.resolve(),
            ensembles=(
                EnsembleSpec(
                    ensemble_id="wild_type_ensemble",
                    replicas=(
                        ReplicaSpec(
                            replica_id="rep_01",
                            trajectory=pdb_path.resolve(),
                            metadata={"condition": "apo", "temperature_k": 300},
                        ),
                    ),
                ),
            ),
        ),
    ),
    labels=(
        LabelSpec(
            label_id="condition_label",
            label_type="categorical",
            level="trajectory",
            source="external_experimental",
            assignments={"hgh_receptor:wild_type_ensemble:rep_01": "apo"},
        ),
    ),
    analysis=AnalysisSpec(
        generic_feature_parameters={"frame_stride": 1, "max_frames": 50},
        plugins=("protein.ca_distances_contacts",),
    ),
)

# Save to study workspace
spec_path = study_path / "project.json"
spec_path.write_text(json.dumps(project.to_dict(), indent=2), encoding="utf-8")
print(f"Written validated project specification to: {spec_path}")


## 3. Starting a Managed Study Run with Full Provenance


In [ ]:
# Begin an auditable run within the study
run_dir, provenance = begin_run(study=study_path, run_id="run-001")

print(f"Run directory created: {run_dir}")
print(f"Run provenance keys:   {list(provenance.keys())}")
print(f"Created at:            {provenance.get('created_at_utc')}")


## 4. Running the Managed Project Workflow

To execute the full workflow through the CLI or Python:
```bash
python -m molscope run --study results/workspace_demo/villin_study --run-id run-001
```
Or with direct project specification:
```bash
python -m molscope run --project results/workspace_demo/villin_study/project.json --out-dir results/workspace_demo/output
```
